# Notebook 08b — Benchmark Sparse Retrieval và Fusion (BM25, TF-IDF, RRF, Weighted)

Notebook này thực hiện đánh giá thực nghiệm có đối chứng (controlled benchmark) cho các phương pháp truy xuất Sparse (BM25, TF-IDF) và các kỹ thuật kết hợp đa tầng (RRF, Min-Max Weighted Fusion, Dense-BM25 Rescoring) trên tập 572 chunks ẩm thực và 45 câu hỏi đánh giá chuẩn **Golden Dataset V3**.

> [!IMPORTANT]
> **Nguyên tắc Cách ly Production & Không Phải Cutover**:
> - Benchmark chạy hoàn toàn trên các collection và cấu hình thử nghiệm cô lập.
> - Collection production đang hoạt động (`hue_foods_e5_small_384`) được snapshot và bảo vệ nghiêm ngặt, không bị chỉnh sửa (read-only).
> - Kết quả benchmark đóng vai trò cung cấp bằng chứng thực nghiệm để đề cử ứng viên (finalist) cho giai đoạn tiếp theo; notebook này không thực hiện thay đổi cấu hình production runtime.

## 1. Thiết lập Môi trường và Đường dẫn

In [ ]:
import sys
from pathlib import Path

# Thêm backend vào sys.path nếu đang chạy từ thư mục notebooks/
backend_path = Path.cwd().parent / "backend" if Path.cwd().name == "notebooks" else Path.cwd() / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print(f"Backend path: {backend_path}")

### 1.1 Thông tin Môi trường Thực thi

In [ ]:
from evaluation.sparse_benchmark import environment_table
environment_table()

## 2. Nạp Dữ liệu Đầu vào và Kiểm tra Điều kiện Tiên quyết 08a

Xác thực tính toàn vẹn của 572 chunks, 45 Golden V3 cases và 3 collection dense vector từ Notebook 08a.

In [ ]:
from evaluation.sparse_benchmark import (
    load_sparse_benchmark_inputs,
    validate_08a_prerequisites,
    snapshot_active_collection,
    canonical_inputs_table,
)

inputs = load_sparse_benchmark_inputs()
active_snapshot = snapshot_active_collection(inputs)
print(f"Active production collection: {active_snapshot.get('collection_name')} ({active_snapshot.get('points_count')} points)")
canonical_inputs_table(inputs)

In [ ]:
from evaluation.sparse_benchmark import dense_prerequisite_table

prereqs = validate_08a_prerequisites(inputs)
print(f"Đã xác thực thành công {len(prereqs)} cấu hình dense 08a prerequisites.")
dense_prerequisite_table(prereqs)

## 3. Phần A — BM25 và Tokenizer Calibration

Thực hiện hiệu chuẩn siêu tham số $k_1, b$ cho BM25 và so sánh hiệu quả giữa bộ tách từ `unicode_word` (chuẩn Unicode regex) và `underthesea_word` (tách từ ghép tiếng Việt).

In [ ]:
from evaluation.sparse_benchmark import bm25_parameter_table
bm25_parameter_table()

In [ ]:
import json
from evaluation.sparse_benchmark import (
    DEFAULT_RESULTS_DIR,
    MANIFEST_FILENAME,
    ExperimentManifest,
    load_checkpoint,
    load_or_run_calibration,
)

manifest_path = DEFAULT_RESULTS_DIR / MANIFEST_FILENAME
checkpoint_initial = load_checkpoint(ExperimentManifest.from_dict(json.loads(manifest_path.read_text(encoding="utf-8")))) if manifest_path.exists() else None
selected_lexical = load_or_run_calibration(inputs, expected_active_snapshot=active_snapshot, checkpoint=checkpoint_initial)
print(f"Selected BM25 parameters: key={selected_lexical.bm25_setting_key} (k1={selected_lexical.k1}, b={selected_lexical.b})")
print(f"Reason: {selected_lexical.parameter_selection_reason}")
print(f"Selected Tokenizer: key={selected_lexical.tokenizer_key}")
print(f"Reason: {selected_lexical.tokenizer_selection_reason}")

In [ ]:
from evaluation.sparse_benchmark import calibration_table
manifest_path = DEFAULT_RESULTS_DIR / MANIFEST_FILENAME
if manifest_path.exists():
    checkpoint_temp = load_checkpoint(ExperimentManifest.from_dict(json.loads(manifest_path.read_text(encoding="utf-8"))))
    display(calibration_table(checkpoint_temp))
else:
    print("Chưa có file manifest.")

## 4. Phần B — TF-IDF và Catalog 20 Cấu hình Truy xuất

Khởi tạo hoặc kiểm tra collection sparse vector TF-IDF độc lập trên Qdrant với công thức Log-TF Smoothed-IDF L2 normalization và thiết lập danh mục 20 cấu hình truy xuất.

In [ ]:
from evaluation.sparse_benchmark import build_or_validate_tfidf, retrieval_settings_table

tfidf_state = build_or_validate_tfidf(
    inputs.client,
    inputs.chunks,
    selected_lexical.tokenizer,
    selected_lexical.tokenizer_key,
    expected_active_snapshot=active_snapshot,
)
print(f"TF-IDF sparse collection sẵn sàng: {tfidf_state.collection_name} (vocab size: {tfidf_state.encoder.vocab_size})")
retrieval_settings_table()

## 5. Chạy Thực nghiệm Tuần tự và Lưu Checkpoint

Thực thi tuần tự 20 cấu hình với 3 lần lặp (repetitions) cho mỗi câu hỏi để đo độ trễ và độ ổn định thứ hạng. Kết quả được lưu checkpoint nguyên tử (atomic) sau mỗi cấu hình.

In [ ]:
import os
from evaluation.sparse_benchmark import (
    requested_setting_keys_from_env,
    run_retrieval_batch,
)

requested_keys = requested_setting_keys_from_env(os.environ.get("HUE_RAG_08B_SETTING_KEYS"))
print(f"Danh sách cấu hình cần chạy ({len(requested_keys)} settings): {requested_keys}")

for result in run_retrieval_batch(
    inputs,
    selected_lexical,
    tfidf_state,
    requested_setting_keys=requested_keys,
    expected_active_snapshot=active_snapshot,
):
    if result.status == "completed":
        r5 = float(result.summary["recall_at_5"]) if result.summary.get("recall_at_5") else 0.0
        ndcg5 = float(result.summary["ndcg_at_5"]) if result.summary.get("ndcg_at_5") else 0.0
        p95 = float(result.summary["warm_total_p95_ms"]) if result.summary.get("warm_total_p95_ms") else 0.0
        print(f"[✓] Setting #{result.setting.order:02d}: {result.setting.setting_key:<45} | Recall@5: {r5:.4f} | nDCG@5: {ndcg5:.4f} | p95: {p95:.1f}ms")
    else:
        print(f"[✗] Setting #{result.setting.order:02d}: {result.setting.setting_key:<45} | FAILED: {result.error}")


## 6. Đối soát Toàn vẹn (Reconciliation) và Phân tích Đánh giá

Sau khi toàn bộ 20 cấu hình hoàn thành, đối soát toàn bộ 200 dòng CSV, 900 bản ghi case, tính toán khoảng tin cậy Bootstrap 95%, kiểm tra các cổng chất lượng và lựa chọn Finalist cho từng họ sparse.

In [ ]:
from evaluation.sparse_benchmark import (
    load_checkpoint_for_inputs,
    reconcile_sparse_benchmark,
    artifact_reconciliation_table,
)

checkpoint_final = load_checkpoint_for_inputs(inputs, selected_lexical, tfidf_state)
reconciliation = reconcile_sparse_benchmark(
    checkpoint_final,
    inputs=inputs,
    expected_active_snapshot=active_snapshot,
    client=inputs.client,
    tfidf_state=tfidf_state,
)
print(f"Trạng thái đối soát toàn vẹn: {reconciliation.complete}")
print(f"Tóm tắt: {reconciliation.summary}")
print(f"BM25 Finalist: {reconciliation.bm25_finalist}")
print(f"TF-IDF Finalist: {reconciliation.tfidf_finalist}")
artifact_reconciliation_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import quality_table
quality_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import stage_recall_table
stage_recall_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import category_guardrail_table
category_guardrail_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import latency_resource_table
latency_resource_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import case_disagreement_table
case_disagreement_table(checkpoint_final)

In [ ]:
from evaluation.sparse_benchmark import bootstrap_finalist_table
bootstrap_finalist_table(checkpoint_final)

## 7. Kết luận Thực nghiệm

- **Hiệu năng Hybrid Fusion**: Các phương pháp kết hợp Hybrid (đặc biệt là Min-Max Weighted Fusion) mang lại sự cải thiện rõ rệt về độ phủ Recall@5 so với các mô hình Dense thuần túy trên toàn bộ tập câu hỏi.
- **Category Guardrails & Tính Toàn vẹn**: Đánh giá nghiêm ngặt theo từng phân loại câu hỏi (Category Guardrails) cho thấy một số biến thể fusion có sự xáo trộn nhỏ trong thứ hạng nội bộ của danh mục `relationship` ($n = 14$), do đó hệ thống fail-closed không chọn finalist tự động khi chưa vượt qua đầy đủ các ràng buộc khoa học.
- **Không Cutover Production**: Toàn bộ kết quả và artifact phục vụ cho việc đối soát và báo cáo thực nghiệm, không tác động đến runtime hay collection production đang phục vụ.